# DNN Universal Approximation — Colab

**Her oturumda:** Hücre 1-4'ü sırayla çalıştır, sonra istediğin bölüme geç.

**Öncesinde:** Runtime > Change runtime type > **GPU** (T4)

---
## 1. Kurulum

In [ ]:
# ── 1.1 Repo sync ────────────────────────────────────────────────
# Öncesinde: Sol panel > Anahtar ikonu (Secrets) > GITHUB_TOKEN ekle
import os, sys
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
REPO  = f'https://{TOKEN}@github.com/barandincoguz/DNN_Project.git'
DIR   = '/content/DNN_Project'

if not os.path.exists(DIR):
    !git clone {REPO} {DIR}
else:
    !cd {DIR} && git pull https://{TOKEN}@github.com/barandincoguz/DNN_Project.git main

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

print('Çalışma dizini:', os.getcwd())

In [ ]:
# ── 1.2 Bağımlılıklar ────────────────────────────────────────────
!pip install -q -r requirements.txt
print('Kurulum tamamlandı.')

In [ ]:
# ── 1.3 Ortam kontrolü ───────────────────────────────────────────
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Yok'

print(f'Python   : {sys.version.split()[0]}')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}')
print(f'GPU      : {gpu_name}')

if device == 'cpu':
    print('\n[UYARI] GPU bulunamadı. Runtime > Change runtime type > GPU seç.')

In [ ]:
# ── 1.4 W&B login ────────────────────────────────────────────────
# Öncesinde: Sol panel > Anahtar ikonu (Secrets) > WANDB_API_KEY ekle
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('WANDB_API_KEY'))
print('W&B login başarılı.')

---
## 2. EDA (Detaylı Veri Analizi)

In [ ]:
# ── 2.1 Genel bakış ──────────────────────────────────────────────
import pandas as pd
import numpy as np

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

print('=== TRAIN ===')
print(f'Shape: {train.shape}')
print(f'\nİlk 5 satır:')
display(train.head())

print('\n=== Eksik değerler ===')
display(train.isnull().sum().to_frame('train_null').join(
    test.isnull().sum().to_frame('test_null')))

print('\n=== İstatistikler ===')
display(train.describe())

In [ ]:
# ── 2.2 Feature dağılımları ──────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

features = ['f1', 'f2', 'f3', 'f4', 'f5']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(train[col].dropna(), bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_title(f'{col} (skew={train[col].skew():.2f})')
    axes[i].set_xlabel(col)

# Target
axes[5].hist(train['target'], bins=40, edgecolor='black', alpha=0.7, color='tomato')
axes[5].set_title(f'target (skew={train["target"].skew():.2f})')

plt.suptitle('Feature Dağılımları', fontsize=14)
plt.tight_layout()
plt.show()
print('[NOT] f5 = tuzak feature, modelde kullanılmayacak.')

In [ ]:
# ── 2.3 Korelasyon ve feature-target ilişkileri ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Korelasyon ısı haritası (f5 dahil — tuzak mı değil mi görelim)
corr = train[['f1','f2','f3','f4','f5','target']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], square=True)
axes[0].set_title('Korelasyon Matrisi')

# Feature-target scatter (kullanılacak 4 feature)
axes[1].axis('off')
fig2, axes2 = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(['f1','f2','f3','f4']):
    axes2[i].scatter(train[col], train['target'], alpha=0.2, s=8, color='steelblue')
    axes2[i].set_xlabel(col)
    axes2[i].set_ylabel('target' if i == 0 else '')
    axes2[i].set_title(f'{col} vs target')

plt.suptitle('Feature-Target İlişkileri', fontsize=13)
plt.tight_layout()
plt.show()

# Özet
print('Kullanılacak featureların target ile korelasyonu:')
print(corr['target'][['f1','f2','f3','f4']].to_string())

In [ ]:
# ── 2.4 f3 log1p dönüşümü etkisi ─────────────────────────────────
f3 = train['f3'].clip(lower=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(f3, bins=40, edgecolor='black', alpha=0.7)
axes[0].set_title(f'f3 orijinal (skew={f3.skew():.2f})')

f3_log = np.log1p(f3)
axes[1].hist(f3_log, bins=40, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title(f'f3 log1p sonrası (skew={f3_log.skew():.2f})')

plt.tight_layout()
plt.show()

---
## 3. Eğitim

In [ ]:
# ── 3.1 Config'i kontrol et / değiştir ───────────────────────────
import src.config as cfg

# Colab'da DEVICE'ı zorla — config.py MPS'i kontrol ediyor, Colab'da yok
# Ama zaten CUDA varsa config.py onu seçer. Kontrol amaçlı print:
print('DEVICE    :', cfg.DEVICE)
print('HIDDEN    :', cfg.HIDDEN_LAYERS)
print('LR        :', cfg.LEARNING_RATE)
print('DROPOUT   :', cfg.DROPOUT_RATE)
print('EPOCHS    :', cfg.EPOCHS)
print('PATIENCE  :', cfg.PATIENCE)
print('LOSS      :', cfg.LOSS_FN)
print('WANDB     :', cfg.WANDB_ENABLED)

# Gerekirse burada override et:
# cfg.LEARNING_RATE = 5e-4
# cfg.HIDDEN_LAYERS = [256, 512, 256, 128]

In [ ]:
# ── 3.2 Eğitimi başlat ───────────────────────────────────────────
# W&B dashboard: https://wandb.ai/[kullanici]/dnn-universal-approx
!python -m src.train

---
## 4. Submission Üret
> `src.predict` eğitimi tekrar çalıştırır + submission + grafikleri kaydeder.

In [ ]:
# ── 4.1 Submission + grafikler üret ─────────────────────────────
# src.predict = train_all_folds() + plot_training_curves() + submission.csv
!python -m src.predict

---
## 5. Sonuçlar

# ── 5.1 Training curves ──────────────────────────────────────────
import os
from PIL import Image

path = 'outputs/plots/training_curves.png'
if os.path.exists(path):
    display(Image.open(path))
else:
    print('[!] Grafik yok — önce 4.1 hücresini çalıştır.')

In [ ]:
# ── 5.2 Residual analizi ─────────────────────────────────────────
path = 'outputs/plots/residual_analysis.png'
if os.path.exists(path):
    display(Image.open(path))
else:
    print('[!] Grafik yok — önce 4.1 hücresini çalıştır.')

In [ ]:
# ── 5.3 Submission önizleme ──────────────────────────────────────
import pandas as pd

sub = pd.read_csv('outputs/submission.csv')
print(f'Shape : {sub.shape}')
print(f'Target: mean={sub.target.mean():.4f}, std={sub.target.std():.4f}')
display(sub.head(10))

In [ ]:
# ── 5.4 Submission indir ─────────────────────────────────────────
from google.colab import files
files.download('outputs/submission.csv')

In [ ]:
# ── 6.1 Sonuçları push et ────────────────────────────────────────
from google.colab import userdata
import os

github_token = userdata.get('GITHUB_TOKEN')
commit_msg   = input('Commit mesajı (örn: mae=0.1234 deeper-arch): ')

os.chdir('/content/DNN_Project')

!git config user.email "colab@dnn"
!git config user.name "Colab"
!git add outputs/submission.csv outputs/plots/ Reports/

# commit: değişiklik yoksa zaten atlar, push her zaman çalışır
!git diff --cached --quiet || git commit -m "{commit_msg}"
!git push https://{github_token}@github.com/barandincoguz/DNN_Project.git main

print('Bitti.')

---
## 7. Optuna Tuning (Opsiyonel)

In [ ]:
# ── 7.1 Hyperparameter arama ─────────────────────────────────────
# Uzun sürer (n_trials * fold sayısı kadar eğitim)
# Sonuç: en iyi config terminale yazdırılır, W&B'ye loglanır
!python -m src.tune